# 从成分记录恢复用量：逆矩阵

甲乙每份 100 克。成分表的每列对应一种原料，每行对应蛋白质、脂肪；数字的含义是每份原料贡献多少克成分。

我们先手推恢复规则，再把它写成矩阵、逐步消元、检查不可逆情况，最后用坐标剪切作对照。

依赖 NumPy、Matplotlib，按顺序运行即可。图采用英文，避免中文字体缺失。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11})
np.set_printoptions(precision=6, suppress=True)
A = np.array([[10., 20.], [20., 10.]])
R = np.array([[-1., 2.], [2., -1.]])/30


## 1. 先正向计算，再从输出恢复

设甲一份、乙两份。先预测两项成分，再运行。

从两项成分 P、F 消元，得到甲 r=(2F-P)/30、乙 s=(2P-F)/30。先把输出代回这两条公式，再用矩阵核对。

把原用量改为 (2,1)、(0.5,1.5)，检查同一条恢复规则是否仍然成立。

In [ ]:
amounts = np.array([1., 2.])  # 再试 [2,1]、[0.5,1.5]
nutrients = A @ amounts
P, F = nutrients
recovered_by_formula = np.array([(2*F-P)/30, (2*P-F)/30])
recovered_by_matrix = R @ nutrients
print("原用量（份）:", amounts)
print("成分（克）:", nutrients)
print("公式恢复（份）:", recovered_by_formula)
print("矩阵恢复（份）:", recovered_by_matrix)
assert np.allclose(recovered_by_formula, amounts)
assert np.allclose(recovered_by_matrix, amounts)


## 2. 对某个目标恢复成功，够了吗？

下面比较 RA 与 AR。一个是“用量→成分→用量”，另一个是“成分→用量→成分”。它们为什么都应该等于单位矩阵，而不只是某个特定向量？

再把 R 的左上角系数由 -1/30 改成 0。观察哪几项检查失败。不要修改真正的 R，下面单独复制了一个候选。

In [ ]:
candidate = R.copy()
# candidate[0,0] = 0  # 取消注释，故意破坏恢复规则
print("先 A 后候选恢复（RA）:\n", candidate @ A)
print("先候选恢复后 A（AR）:\n", A @ candidate)
print("两方向都是单位矩阵:",
      np.allclose(candidate @ A, np.eye(2)) and
      np.allclose(A @ candidate, np.eye(2)))

elementwise_reciprocal = 1/A
print("逐项倒数得到的用量:", elementwise_reciprocal @ np.array([50.,40.]))
print("真正的逆恢复的用量:", R @ np.array([50.,40.]))
assert not np.allclose(elementwise_reciprocal @ A, np.eye(2))
assert np.allclose(R @ A, np.eye(2))
assert np.allclose(A @ R, np.eye(2))


## 3. 先给目标，再检查是不是可用配方

输入 (50,40) 克、(40,50) 克；再试 (10,0) 克。三个目标都有唯一实数输入吗？第三个目标为什么不能实际配出来？

注意逆矩阵负责恢复实数输入，不负责保证用量非负。

In [ ]:
target = np.array([10., 0.])  # 再试 [50,40]、[40,50]
solution = R @ target
print("目标成分（克）:", target)
print("唯一实数用量（份）:", solution)
print("成分重建（克）:", A @ solution)
print("用量都非负:", bool(np.all(solution >= -1e-10)))
assert np.allclose(A @ solution, target)


## 4. 逆的第一列，到底在回答什么？

第一列要恢复目标 (1,0)，第二列要恢复目标 (0,1)。先分别手算这两组方程，再看下面输出。

把任意目标写成 P(1,0)+F(0,1)，为什么对应的输入也能按相同系数组合？

In [ ]:
r1, r2 = R[:,0], R[:,1]
print("第一列输入（份）:", r1, "输出（克）:", A @ r1)
print("第二列输入（份）:", r2, "输出（克）:", A @ r2)
assert np.allclose(A @ r1, [1,0])
assert np.allclose(A @ r2, [0,1])
P, F = 50., 40.
combined_input = P*r1+F*r2
print("组合输入（份）:", combined_input)
print("组合输出（克）:", A @ combined_input)
assert np.allclose(A @ combined_input, [P,F])


## 5. 不调用求逆函数，逐步处理 [A|I]

右边的两列是刚才的两个基本目标，不是附加约束。每一步行操作同时作用于整行，不能只修改左半边。

在运行前，手算每一步右边的变化，特别检查最后的 1/10-2/15。

In [ ]:
augmented = np.column_stack([A, np.eye(2)])
steps = [("开始 [A|I]", augmented.copy())]
augmented[0] /= 10
steps.append(("第1行除以10", augmented.copy()))
augmented[1] -= 20*augmented[0]
steps.append(("第2行减去第1行的20倍", augmented.copy()))
augmented[1] /= -30
steps.append(("第2行除以-30", augmented.copy()))
augmented[0] -= 2*augmented[1]
steps.append(("第1行减去第2行的2倍", augmented.copy()))
for label, matrix in steps:
    print(label)
    print(matrix)
left, recovered_inverse = augmented[:,:2], augmented[:,2:]
assert np.allclose(left, np.eye(2))
assert np.allclose(recovered_inverse, R)
# 这里只用现成函数作最后核对，不用它代替推导。
assert np.allclose(recovered_inverse, np.linalg.inv(A))


## 6. 改变成分表，恢复规则何时失效？

把乙改成甲的两倍：甲仍为 (10,20)，乙变为 (20,40)。

先预测甲两份与乙一份的成分。只有输出时，能判断原来是哪种用量吗？再判断目标 (50,40) 为什么根本到不了。

下面的函数失败只是数值核对；不能有逆的理由是两个不同输入合并为同一个输出。

In [ ]:
C = np.array([[10.,20.], [20.,40.]])
first, second = np.array([2.,0.]), np.array([0.,1.])
print("甲两份的成分:", C @ first)
print("乙一份的成分:", C @ second)
print("两个不同输入的输出相同:", np.allclose(C @ first, C @ second))
assert not np.allclose(first, second)
assert np.allclose(C @ first, C @ second)
print("无法达到 (50,40)：所有输出都要求脂肪=2×蛋白质。")
try:
    np.linalg.inv(C)
except np.linalg.LinAlgError:
    print("核对：这张成分表没有逆矩阵。")
else:
    raise AssertionError("这里应该是不可逆矩阵")


## 7. 换成坐标剪切

位置 (s,t) 变成 (s+t,t)。从输出 (30,10) 恢复输入，先看第二项，再从第一项减掉它。

图显示连续的坐标变换，不涉及裁剪、像素取整或压缩。它们若额外丢失信息，恢复坐标不等于恢复全部图像数据。

In [ ]:
S = np.array([[1.,1.], [0.,1.]])
S_inverse = np.array([[1.,-1.], [0.,1.]])
print("输出(30,10)的原位置:", S_inverse @ np.array([30.,10.]))
assert np.allclose(S_inverse @ S, np.eye(2))
assert np.allclose(S @ S_inverse, np.eye(2))

fig, axes = plt.subplots(1, 3, figsize=(12, 4), dpi=150)
point = np.array([1.,1.])
for ax, matrix, title in zip(axes, [np.eye(2), S, S_inverse @ S],
                              ["Input", "Sheared", "Restored"]):
    for coordinate in range(-2, 3):
        for line in (np.array([[-2, coordinate], [2, coordinate]]),
                     np.array([[coordinate, -2], [coordinate, 2]])):
            transformed = line @ matrix.T
            ax.plot(transformed[:,0], transformed[:,1], color="#2878b5", alpha=0.5)
    transformed_point = matrix @ point
    ax.scatter(*transformed_point, color="#d97922", s=45, zorder=5)
    ax.set(xlim=(-4.5,4.5), ylim=(-3,3), aspect="equal", title=title,
           xlabel="Coordinate 1", ylabel="Coordinate 2")
    ax.grid(alpha=0.15)
fig.tight_layout()
plt.show()


## 8. 先剪切后拉伸，恢复时从哪步开始？

输入 (2,1)。先剪切，再把高度乘 2。先手算中间位置与最终位置，然后比较两种撤销顺序。

把输入改成别的点检查。某些特殊点可能两种顺序恰好都恢复，为什么不能用单点成功证明规则正确？

In [ ]:
D = np.diag([1.,2.])
D_inverse = np.diag([1.,0.5])
point = np.array([2.,1.])  # 也试 [0,0]、[2,0]、[1,3]
after_shear = S @ point
output = D @ after_shear
correct = S_inverse @ (D_inverse @ output)
wrong = D_inverse @ (S_inverse @ output)
print("输入:", point, "剪切后:", after_shear, "拉伸后:", output)
print("先撤销拉伸，再撤销剪切:", correct)
print("先撤销剪切，再撤销拉伸:", wrong)
assert np.allclose(correct, point)
correct_rule = S_inverse @ D_inverse
wrong_rule = D_inverse @ S_inverse
assert np.allclose(correct_rule @ (D @ S), np.eye(2))
assert not np.allclose(wrong_rule @ (D @ S), np.eye(2))


## 合上实验，自己解释

1. 从两条成分方程推导通用恢复公式，再写成逆矩阵。
2. 分别解释 RA=I 与 AR=I 的输入输出含义。
3. 为什么求逆的两列就是解两个标准基目标？[A|I] 的右边在记录什么？
4. 逐项取倒数为什么不是逆矩阵？
5. 用两个合并的输入说明逆不存在，不要只说“函数报错”。
6. 逆能恢复实数关系，为什么仍需检查非负用量？
7. 先A后B，为什么必须先撤销B？先用具体数字说清，再证明乘积公式。